# Duyusal Sinek: NULL-CONNECTOME kontrollü versiyon

**2026-09-22.** Gemini'nin `Benchmark_Karsilastirma_Colab.ipynb` / `Mega_Biyolojik_Turnuva.ipynb` notebook'ları, sinek beynine hedef fonksiyonun gerçek matematiksel gradyanını besleyip PSO'ya (gradyansız arama) karşı yarıştırmıştı. Bu, connectome yapısından değil gradyan-enjeksiyonundan kaynaklanan bir avantaj — hemen hemen HER ağ (gerçek, null, hatta rastgele) bu kurguda PSO'yu yenerdi, bu yüzden FlyOpt projesinin asıl sorusuna (**gerçek connectome vs null connectome**) hiçbir kanıt sağlamıyordu.

Bu notebook AYNI ajan tasarımını (gradyan enjeksiyonu + T=20 biyolojik motor) korur, ama kıyaslama eksenini düzeltir: PSO yerine **aynı alt-grafın degree_preserving_rewire null'una karşı**, n=8 bağımsız tohumla (yeniden-kablolama + sürü başlangıcı), bu projenin standart istatistiğiyle (Wilcoxon, Mann-Whitney, Cliff's δ) test eder.

Her iki kol da (gerçek ve null) eşit derecede **eğitilmemiş** (rastgele başlangıç ağırlıklı) — orijinal notebook'larla tutarlı (hiçbirinde `train()` çağrısı yoktu).

In [ ]:
# 1. GEREKSİNİMLER VE DRIVE BAĞLANTISI
from google.colab import drive
import sys, os, json
import torch
import numpy as np
import torch.nn.functional as F
from scipy import sparse, stats

drive.mount('/content/drive')
project_path = '/content/drive/MyDrive/fly_op'
if not os.path.exists(project_path):
    project_path = '/content/drive/MyDrive/fly_op/fly_op'
sys.path.append(project_path)
sys.path.append(os.path.join(project_path, 'src'))
os.chdir(project_path)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Aktif donanim: {DEVICE}')

In [ ]:
# 2. ALGORİTMA VE AĞ TANIMLARI
import glob
from flyopt.substrates.graph_builders import degree_preserving_rewire
from flyopt.variants.rate_brain import RateBrain, RateBrainConfig, build_subgraph_bfs, select_connected_encode_decode

DIM = 2
N_ENCODE = 4
N_DECODE = 4
SUBGRAPH_SIZE = 3000
T = 20
NUM_AGENTS = 2000
ITERS = 100
STEP_SCALE = 0.1
SEEDS = list(range(8))

def sphere(x):
    return torch.sum(x**2, dim=1)

def rastrigin(x):
    return 10 * x.shape[1] + torch.sum(x**2 - 10 * torch.cos(2 * np.pi * x), dim=1)

def sensory_fly_optimize(func, brain, num_particles, iters, step_scale, seed):
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    x = (torch.rand((num_particles, DIM), device=DEVICE, generator=gen) * 10 - 5)
    x.requires_grad_(True)
    best_obj = func(x).detach()
    for _ in range(iters):
        obj = func(x)
        obj.sum().backward()
        with torch.no_grad():
            grad = x.grad.clone()
            x.grad.zero_()
            sensory = F.normalize(-grad, p=2, dim=1)
            pad = torch.zeros((x.shape[0], N_ENCODE - DIM), device=DEVICE)
            env_input = torch.cat([sensory, pad], dim=1)
            move = brain(env_input)[:, :DIM] * step_scale
            x_new = (x + move).detach()
            new_obj = func(x_new)
            mask = new_obj < best_obj
            x.data[mask] = x_new[mask]
            best_obj[mask] = new_obj[mask]
    return float(best_obj.min().item())

print('tanimlar hazir')

In [ ]:
# 3. MALE CNS AĞINI YÜKLE
found_files = glob.glob('/content/drive/MyDrive/**/malecns_adjacency.npz', recursive=True)
DATA_PROCESSED = os.path.dirname(found_files[0]) if found_files else f"{project_path}/data/processed"
print('veri konumu:', DATA_PROCESSED)

base_weights = sparse.load_npz(f"{DATA_PROCESSED}/malecns_adjacency.npz")
afferent = np.load(f"{DATA_PROCESSED}/malecns_afferent_indices.npy")
efferent = np.load(f"{DATA_PROCESSED}/malecns_efferent_indices.npy")

encode_full, decode_full = select_connected_encode_decode(
    base_weights, afferent, efferent, n_encode=N_ENCODE, n_decode=N_DECODE,
    max_hops=6, n_encode_candidates=200, seed=9000,
)
sub_real, encode_idx, decode_idx, _ = build_subgraph_bfs(base_weights, encode_full, decode_full, SUBGRAPH_SIZE, seed=9000)
print(f"male CNS subgraph: {sub_real.shape[0]} nodes, {sub_real.nnz} edges, {len(encode_idx)} encode / {len(decode_idx)} decode")

In [ ]:
# 4. GERCEK vs NULL, N=8 TOHUM, HER GOREV ICIN
def run_task(task_name, func):
    cfg = RateBrainConfig(dim=DIM, n_readout=len(decode_idx), T=T, decode_scale=0.5, train_gain=True)
    real_list, null_list = [], []
    for seed in SEEDS:
        sub_null = degree_preserving_rewire(sub_real, seed=seed)
        for lst, sub in [(real_list, sub_real), (null_list, sub_null)]:
            brain = RateBrain(sub, encode_idx, decode_idx, cfg, seed=seed).to(DEVICE)
            final = sensory_fly_optimize(func, brain, NUM_AGENTS, ITERS, STEP_SCALE, seed=seed)
            lst.append(final)
        print(f"  [{task_name}] seed={seed}: real={real_list[-1]:.5f}  null={null_list[-1]:.5f}")

    real, null = np.array(real_list), np.array(null_list)
    w_p = stats.wilcoxon(real, null).pvalue
    mw_p = stats.mannwhitneyu(real, null, alternative="two-sided").pvalue
    gt = np.sum(real[:, None] < null[None, :]); lt = np.sum(real[:, None] > null[None, :])
    delta = float((gt - lt) / (len(real) * len(null)))
    gate = bool(mw_p < 0.05 and abs(delta) > 0.33)
    print(f"\n=== {task_name}: real_median={np.median(real):.5f} null_median={np.median(null):.5f} "
          f"Wilcoxon p={w_p:.4f} Mann-Whitney p={mw_p:.4f} delta={delta:.3f} gate={'PASS' if gate else 'FAIL'} ===\n")
    return {"real": real.tolist(), "null": null.tolist(), "wilcoxon_p": float(w_p),
            "mannwhitney_p": float(mw_p), "cliffs_delta": delta, "gate_pass": gate}

results = {}
results["sphere"] = run_task("sphere", sphere)
results["rastrigin"] = run_task("rastrigin", rastrigin)

with open("sensory_fly_nullcontrol_summary.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk not in ('real','null')} for k, v in results.items()}, indent=2))

In [ ]:
# 5. INDIR
from google.colab import files
files.download("sensory_fly_nullcontrol_summary.json")